# 03 — Train YOLO11n (**two classes**: healthy / unhealthy)

Trains an Ultralytics **YOLO11n** detector on the 2-class boxes produced by `01_dataset_prep`. All knobs live in the **config cell** below — nothing is buried in code.

Prereqs: run `01_dataset_prep` first (it writes `croprow_disease/data/health.yaml` and the `labels_health/` tree). Needs the torch + ultralytics stack; a CUDA GPU is strongly recommended.

> **Check the class-balance gate before you train.** On LettuceMOTS alone the `unhealthy` class is empty — training here produces a model that calls everything healthy and still reports a high mAP. The cell below refuses to start in that case. The real path to a working model on this dataset is `04_finetune` against frames that actually contain affected plants.

> Ships **unrun** on purpose — whoever trains runs it top to bottom.

In [ ]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow_disease/utils.py) so the package imports
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow_disease" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow_disease import utils as U
from croprow_disease.health import HealthParams

CW = REPO_ROOT / "croprow_disease"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

# ===================== CONFIG (edit here only) =====================
MODEL      = "yolo11n.pt"          # base weights (auto-downloaded by ultralytics)
DATA_YAML  = str(DATA_DIR / "health.yaml")

IMGSZ      = 640
EPOCHS     = 100
BATCH      = 16                    # -1 for auto-batch on GPU
LR0        = 0.01                  # initial learning rate
LRF        = 0.01                  # final LR = LR0 * LRF (cosine)
OPTIMIZER  = "auto"                # auto | SGD | Adam | AdamW
PATIENCE   = 30                    # early-stop patience (epochs)
WORKERS    = 8
DEVICE     = 0                     # GPU index, or "cpu"
SEED       = 42

# The two classes are heavily imbalanced in most field data (healthy plants far
# outnumber affected ones). Ultralytics has no class-weighting knob, so the
# lever that matters is the data itself -- see 04. Leave this True to have the
# notebook stop rather than train a degenerate model.
REQUIRE_BALANCED = True

RUN_NAME   = f"train_yolo11n_health_{IMGSZ}"
# ===================================================================
print("data yaml:", DATA_YAML)
print("run ->", RUNS_DIR / RUN_NAME)

## Environment check

In [ ]:
# This notebook needs the training/inference stack (torch + ultralytics),
# NOT installed in the light 01/02 env. See croprow_disease/requirements-train.txt.
try:
    import torch
    from ultralytics import YOLO
    import ultralytics
    print("torch      :", torch.__version__)
    print("ultralytics:", ultralytics.__version__)
    print("CUDA avail :", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"Missing training dependency: {e.name}. Install "
        "croprow_disease/requirements-train.txt into the croprow Python 3.11 venv "
        "before running this notebook."
    ) from e

## Guard: is the training set actually two-class?

Counts the labels the trainer is about to read. This is the cell that stops you shipping a model whose `unhealthy` head never saw a positive example.

In [ ]:
import collections

if not Path(DATA_YAML).is_file():
    raise FileNotFoundError(
        f"{DATA_YAML} not found. Run 01_dataset_prep first.")

import yaml as _yaml
cfg = _yaml.safe_load(Path(DATA_YAML).read_text())
print("classes:", cfg["names"], "| nc =", cfg["nc"])

counts = collections.Counter()
for split in ("train", "val"):
    list_file = Path(cfg[split])
    imgs = [l for l in list_file.read_text().splitlines() if l.strip()]
    for img in imgs:
        lab = Path(str(Path(img)).replace(f"{os.sep}images{os.sep}",
                                          f"{os.sep}{U.LABEL_SUBDIR}{os.sep}")
                   ).with_suffix(".txt")
        if not lab.is_file():
            continue
        for line in lab.read_text().splitlines():
            if line.strip():
                counts[U.CLASS_NAMES[int(line.split()[0])]] += 1
    print(f"  {split}: {len(imgs)} images")

verdict = U.check_class_balance(counts)
print()
print(verdict["message"])

if REQUIRE_BALANCED and not verdict["ok"]:
    raise RuntimeError(
        "Refusing to train: see the class-balance verdict above. Set "
        "REQUIRE_BALANCED = False to override deliberately (and then do NOT "
        "quote the resulting mAP as disease-detection accuracy).")

## Train

In [ ]:
model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    lr0=LR0,
    lrf=LRF,
    optimizer=OPTIMIZER,
    patience=PATIENCE,
    workers=WORKERS,
    device=DEVICE,
    seed=SEED,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)
save_dir = Path(model.trainer.save_dir)
print("run dir:", save_dir)

## Validate best weights + copy them into croprow_disease/models/

In [ ]:
import shutil
best = save_dir / "weights" / "best.pt"
print("best weights:", best, "exists:", best.is_file())

MODELS_DIR.mkdir(parents=True, exist_ok=True)
dst = MODELS_DIR / "best.pt"
if best.is_file():
    shutil.copy2(best, dst)
    print("copied ->", dst)

m = YOLO(str(best)).val(data=DATA_YAML, imgsz=IMGSZ, device=DEVICE, split="val")
map50, map5095 = float(m.box.map50), float(m.box.map)
precision, recall = float(m.box.mp), float(m.box.mr)

# Per-class mAP50. m.box.maps is indexed by class id.
per_class = {U.CLASS_NAMES[i]: float(v) for i, v in enumerate(m.box.maps)}
print(f"mAP50={map50:.4f}  mAP50-95={map5095:.4f}  P={precision:.4f}  R={recall:.4f}")
for name, v in per_class.items():
    print(f"  {name:10s} mAP50-95: {v:.4f}")

## Log the run to RESULTS.md

`dataset` is `LettuceMOTS-val`; `labels` records that the classes are colour-derived. Public metrics are logged as their own row — never merged with own-frame results.

In [ ]:
U.append_results_row(
    RESULTS_MD,
    run=RUN_NAME,
    dataset="LettuceMOTS-val",
    labels="colour-derived",
    map50=map50, map5095=map5095,
    precision=precision, recall=recall,
    map50_healthy=per_class.get("healthy", "-"),
    map50_unhealthy=per_class.get("unhealthy", "-"),
    epochs=EPOCHS, imgsz=IMGSZ,
)
print(RESULTS_MD.read_text())